# Execução do TrashNet no Google Colab — sem Google Drive

Este notebook utiliza o ZIP oficial já versionado no repositório GitHub. **Não é necessário montar o Google Drive.**

1. Selecione GPU em **Ambiente de execução → Alterar o tipo de ambiente de execução**.
2. Execute as células na ordem.
3. Faça backups dos resultados para o computador ao terminar cada etapa importante.

**Atenção:** o projeto está dentro da subpasta `trashnet_mlp_cnn_unifor/` do repositório. A configuração abaixo já considera essa organização. Os arquivos em `/content` são temporários e podem ser perdidos quando a sessão Colab terminar.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/felipecarvalho98-bit/trashnet-mlp-cnn-unifor.git"
REPO_DIR = Path("/content/trashnet-mlp-cnn-unifor")
PROJECT_DIR = REPO_DIR / "trashnet_mlp_cnn_unifor"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)

if not (PROJECT_DIR / "src" / "prepare_data.py").exists():
    raise FileNotFoundError(f"Não encontrei o código do projeto em {PROJECT_DIR}")

LOCAL_DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
os.environ["TRASHNET_DATA_DIR"] = str(LOCAL_DATA_DIR)
os.environ["TRASHNET_OUTPUT_DIR"] = str(OUTPUT_DIR)
os.chdir(PROJECT_DIR)

dataset_zip = LOCAL_DATA_DIR / "dataset-resized.zip"
if not dataset_zip.is_file():
    raise FileNotFoundError(f"Dataset não encontrado: {dataset_zip}")

print("Repositório:", REPO_DIR)
print("Pasta do projeto:", PROJECT_DIR)
print("Dataset encontrado:", dataset_zip)
print(f"Tamanho do ZIP: {dataset_zip.stat().st_size / (1024**2):.1f} MiB")
print("Resultados temporários:", OUTPUT_DIR)

## 1. Instalar dependências

O Colab normalmente já inclui TensorFlow. Instalamos as outras bibliotecas sem forçar a reinstalação do TensorFlow fornecido pelo Colab.

In [ ]:
%pip install -q scikit-learn Pillow matplotlib psutil pandas

## 2. Verificar TensorFlow e GPU

Não avance para os treinamentos até confirmar que o TensorFlow detectou a GPU esperada. A disponibilidade e o modelo da GPU dependem da cota e dos recursos atuais do Colab.

In [ ]:
import tensorflow as tf
import platform
import subprocess

gpus = tf.config.list_physical_devices("GPU")
print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPUs detectadas:", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print("Detalhes da GPU:", tf.config.experimental.get_device_details(gpu))
    except Exception as exc:
        print("Aviso ao configurar memory growth:", exc)

if not gpus:
    print("ATENÇÃO: nenhuma GPU foi detectada. Selecione GPU nas configurações do Colab e reinicie o ambiente.")
else:
    with tf.device("/GPU:0"):
        teste = tf.linalg.matmul(
            tf.random.normal((1024, 1024)),
            tf.random.normal((1024, 1024))
        )
    print("Operação de teste executada em:", teste.device)

try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi não está disponível neste runtime.")

## 3. Conferir o ZIP e preparar os dados

O ZIP veio do repositório da equipe. Antes de extrair, verificamos seu SHA-256. Depois, a análise exploratória confere as 2.527 imagens, as seis classes, as dimensões, a integridade e a divisão estratificada 70%/15%/15%.

In [ ]:
from src.common import DATASET_SHA256, DATASET_ZIP, sha256_file

hash_real = sha256_file(DATASET_ZIP)
print("SHA-256 esperado:", DATASET_SHA256)
print("SHA-256 encontrado:", hash_real)
if hash_real != DATASET_SHA256:
    raise ValueError("O ZIP do repositório não corresponde ao arquivo oficial esperado.")

print("Hash confirmado. Iniciando inspeção e preparação...")


In [ ]:
!python -m src.prepare_data

## 4. Conferir o relatório da divisão

Só prossiga se o relatório confirmar 2.527 imagens e as quantidades oficiais por classe. A divisão esperada é 1.768 imagens para treino, 379 para validação e 380 para teste.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Image

inspection = json.loads(Path("outputs/metricas/inspecao_dataset.json").read_text(encoding="utf-8"))
splits = json.loads(Path("outputs/splits.json").read_text(encoding="utf-8"))

print("Total de imagens válidas:", inspection["total_valid_images"])
print("Contagem por classe:", inspection["counts_by_class"])
print("Dimensões:", inspection["dimensions_width_x_height"])
print("Imagens corrompidas:", len(inspection["corrupted_files"]))
print("Grupos duplicados:", len(inspection["exact_pixel_duplicates"]))
display(pd.DataFrame(splits["counts_by_split_and_class"]).T)

for name in ["fig1_distribuicao.png", "fig2_exemplos.png", "fig3_resolucoes.png", "fig4_pesos_classe.png"]:
    display(Image(filename=f"outputs/figuras/{name}"))


## 5. Testar o experimento E3

O E3 é uma CNN de 128 × 96 sem pesos de classe. Este comando executa o treinamento completo, com early stopping — não é apenas um teste de uma época.

In [ ]:
!python -m src.train --exp E3

## 6. Executar os seis experimentos

Depois que E3 terminar sem erros, execute todos. A opção `--skip-existing` evita repetir experimentos que já tenham modelo e configuração salvos na VM atual.

In [ ]:
!python -m src.train --all --skip-existing

## 7. Avaliar os modelos

A avaliação seleciona o modelo de referência pelo F1 macro de validação e só depois calcula as métricas de teste dos seis experimentos.

In [ ]:
from pathlib import Path
from src.common import EXPERIMENTS, MODELS_DIR, METRICS_DIR, HISTORY_DIR
import json

missing = []
for exp in EXPERIMENTS:
    expected = [
        MODELS_DIR / f"modelo_{exp}.keras",
        METRICS_DIR / f"config_{exp}.json",
        HISTORY_DIR / f"historico_{exp}.json",
    ]
    absent = [p.name for p in expected if not p.is_file()]
    status = "OK" if not absent else "FALTANDO: " + ", ".join(absent)
    print(f"{exp}: {status}")
    if absent:
        missing.append(exp)

if missing:
    print("\nAinda faltam treinamentos. Execute a célula anterior (!python -m src.train --all --skip-existing) e depois confira novamente.")
else:
    print("\nOs seis modelos, configurações e históricos estão presentes. Pode executar a avaliação.")

# Resumo do histórico da MLP E1 para investigar desempenho baixo, se disponível.
history_e1 = HISTORY_DIR / "historico_E1.json"
if history_e1.is_file():
    h = json.loads(history_e1.read_text(encoding="utf-8"))
    for metric in ["loss", "accuracy", "val_loss", "val_accuracy"]:
        values = h.get(metric, [])
        if values:
            best = min(values) if "loss" in metric else max(values)
            print(f"E1 {metric}: épocas={len(values)}, final={values[-1]:.4f}, melhor={best:.4f}")


In [ ]:
!python -m src.evaluate

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

csv_files = [\"tabela5_custo.csv\", \"tabela6_desempenho.csv\", \"tabela7_por_classe.csv\"]
missing_csv = [name for name in csv_files if not Path(f\"outputs/metricas/{name}\").is_file()]
if missing_csv:
    print(\"As tabelas ainda não existem. A avaliação não terminou corretamente. Execute os seis treinamentos e rode !python -m src.evaluate antes de tentar exibi-las.\")
    print(\"Arquivos ausentes:\", missing_csv)
else:
    for filename in csv_files:
        print(\"\\n\", filename)
        display(pd.read_csv(f\"outputs/metricas/{filename}\"))

figure_files = [\"fig5_curvas_aprendizagem.png\", \"fig6_matrizes_confusao.png\", \"fig7_resolucao_e_custo.png\", \"fig8_erros_modelo_referencia.png\"]
missing_figures = [name for name in figure_files if not Path(f\"outputs/figuras/{name}\").is_file()]
if missing_figures:
    print(\"Figuras finais ainda não geradas:\", missing_figures)
else:
    for name in figure_files:
        display(Image(filename=f\"outputs/figuras/{name}\"))


## 8. Fazer backup dos resultados

O Google Drive não está montado. Os modelos, checkpoints, históricos, métricas e figuras ficam em `/content`, que é temporário. Execute esta célula depois de cada etapa importante e baixe o ZIP para o computador. Se os arquivos forem grandes, faça backups separados ao longo do projeto.

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

backup_base = Path("/content/TrashNet_outputs_backup")
backup_zip = Path(str(backup_base) + ".zip")
if backup_zip.exists():
    backup_zip.unlink()

shutil.make_archive(str(backup_base), "zip", root_dir=PROJECT_DIR, base_dir="outputs")
print(f"Backup criado: {backup_zip} ({backup_zip.stat().st_size / (1024**2):.1f} MiB)")
files.download(str(backup_zip))
